<a href="https://colab.research.google.com/github/drameyjoshi/tc/blob/main/CH4_molecule.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyscf

In [ ]:
import pyscf
import numpy as np

HARTREE_TO_EV = 27.211386

# Build the molecule

Got the coordinates from [Computational Chemistry Comparison and Benchmark DataBase](https://cccbdb.nist.gov/exp2x.asp).

In [ ]:
mol = pyscf.gto.Mole()
mol.atom = """
C 0 0 0;
H 0.6276 0.6276 0.6276;
H 0.6276 -0.6276 -0.6276;
H -0.6276 0.6276 -0.6276;
H -0.6276 -0.6276 0.6276;
"""
mol.basis = "sto-3g"
mol.verbose = 3
mol.build()

# Choice of algorithm
The total number of electrons is $6 + 4 \times 1 = 10$. All electrons in the molecule will be paired and we can use the restricted Hartree-Fock method.

In [ ]:
# 1. Run Restricted Hartree-Fock (RHF)
print("=== Running RHF ===")
rhf_mol = pyscf.scf.RHF(mol)
rhf_energy = rhf_mol.kernel()

# 2. Run Unrestricted Hartree-Fock (UHF)
print("\n=== Running UHF ===")
uhf_mol = pyscf.scf.UHF(mol)
uhf_energy = uhf_mol.kernel()

# 3. Check Spin Contamination <S^2> for UHF
s2, multip = uhf_mol.spin_square()

print("\n=== Comparison ===")
print(f"RHF Energy: {rhf_energy:.8f} Eh")
print(f"UHF Energy: {uhf_energy:.8f} Eh")
print(f"UHF <S^2> Spin Contamination: {s2:.4f} (0.0000 means it is a pure closed-shell singlet)")

=== Running RHF ===
converged SCF energy = -39.7268091715447

=== Running UHF ===
converged SCF energy = -39.7268091715448  <S^2> = 7.1054274e-15  2S+1 = 1

=== Comparison ===
RHF Energy: -39.72680917 Eh
UHF Energy: -39.72680917 Eh
UHF <S^2> Spin Contamination: 0.0000 (0.0000 means it is a pure closed-shell singlet)


# Diagnostic information
The list of atomic orbitals is $1s, 2s, 2p_x, 2p_y, 2p_z$ for C and four $1s$ for H atoms. Thus, the minimal basis will have nine orbitals.

The molecular orbitals diagonalise the Fock operator. They are the canonical orbitals, eigenvectors of the Fock matrix. However, they are no all alike. They are not the equivalent $sp^3$ hybrid orbitals that we expect from elementary chemistry.

In [29]:
print(f"Elements in molecule: {mol.elements}")
print(f"Number of electrons: {mol.nelectron}")
print(f"Basis functions count: {mol.nao}")

F_ao = rhf_mol.get_fock()
print(f"Fock matrix shape: {F_ao.shape}")
C_val_coeff = rhf_mol.mo_coeff[:, 1:5]
C_val_energy = rhf_mol.mo_energy[1:5]
print(f"Energy of valence orbitals: \n{C_val_energy}")

F_val_can = C_val_coeff.T @ F_ao @ C_val_coeff * HARTREE_TO_EV
print("=== Canonical Valence Fock Matrix (eV) ===")
print(np.round(F_val_can, 6))

Elements in molecule: ['C', 'H', 'H', 'H', 'H']
Number of electrons: 10
Basis functions count: 9
Fock matrix shape: (9, 9)
Energy of valence orbitals: 
[-0.91001197 -0.51890449 -0.51890449 -0.51890449]
=== Canonical Valence Fock Matrix (eV) ===
[[-24.762688   0.         0.         0.      ]
 [  0.       -14.120111   0.         0.      ]
 [  0.         0.       -14.120111   0.      ]
 [  0.         0.         0.       -14.120111]]


## Build localised orbitals

The Fock matrix in localised orbitals has the same value along the diagonal. The remaining components too are quite similar. That is, the four orbitals are indeed of the same magnitude only pointing in different directions.

In [ ]:
localiser = pyscf.lo.Boys(mol, C_val_coeff)
loc_mol = localiser.kernel()

F_val_loc = loc_mol.T @ F_ao @ loc_mol * HARTREE_TO_EV
print("\n=== Localized Valence Fock Matrix (eV) ===")
print(np.round(F_val_loc, 6))

# 5. Inspect AO composition of the localized orbitals
print("\n=== Localized Orbital Coefficients (AO basis) ===")
ao_labels = mol.ao_labels()
for ao_idx, label in enumerate(ao_labels):
    coeffs_str = "  ".join(
        f"{loc_mol[ao_idx, mo_idx]:8.4f}" for mo_idx in range(4)
    )
    print(f"{label:<12}: {coeffs_str}")


WARN: Initial orbitals are close to convergence. Adding a small perturbation.


=== Localized Valence Fock Matrix (eV) ===
[[-16.780759   2.660648  -2.660644  -2.660645]
 [  2.660648 -16.780758   2.660643   2.660645]
 [ -2.660644   2.660643 -16.78075   -2.660641]
 [ -2.660645   2.660645  -2.660641 -16.780753]]

=== Localized Orbital Coefficients (AO basis) ===
0 C 1s      :  -0.1106    0.1106   -0.1106   -0.1106
0 C 2s      :   0.3144   -0.3144    0.3144    0.3144
0 C 2px     :   0.2857    0.2857   -0.2857    0.2857
0 C 2py     :  -0.2857    0.2857    0.2857    0.2857
0 C 2pz     :  -0.2857   -0.2857   -0.2857    0.2857
1 H 1s      :  -0.0603    0.0603   -0.0603    0.5422
2 H 1s      :   0.5422    0.0603   -0.0603   -0.0603
3 H 1s      :  -0.0603    0.0603    0.5422   -0.0603
4 H 1s      :  -0.0603   -0.5422   -0.0603   -0.0603


# Molecular orbital energies

In [ ]:
# Print the molecular orbital (MO) energies using the RHF model object
print("=== Molecular Orbital Energies (Hartrees) ===")
for i, ev in enumerate(rhf_mol.mo_energy):
    occ = rhf_mol.mo_occ[i]
    print(f"Orbital {i+1:2d}: Energy = {ev:11.6f} Hartree, {ev * HARTREE_TO_EV: 11.6f} | Occupation = {occ}")

=== Molecular Orbital Energies (Hartrees) ===
Orbital  1: Energy =  -11.030034 Hartree, -300.142513 | Occupation = 2.0
Orbital  2: Energy =   -0.910012 Hartree,  -24.762687 | Occupation = 2.0
Orbital  3: Energy =   -0.518904 Hartree,  -14.120110 | Occupation = 2.0
Orbital  4: Energy =   -0.518904 Hartree,  -14.120110 | Occupation = 2.0
Orbital  5: Energy =   -0.518904 Hartree,  -14.120110 | Occupation = 2.0
Orbital  6: Energy =    0.715644 Hartree,   19.473656 | Occupation = 0.0
Orbital  7: Energy =    0.715644 Hartree,   19.473656 | Occupation = 0.0
Orbital  8: Energy =    0.715644 Hartree,   19.473656 | Occupation = 0.0
Orbital  9: Energy =    0.754958 Hartree,   20.543466 | Occupation = 0.0


# Ionisation energy

Using minimal basis.

In [ ]:
sto_3g_ie = -rhf_mol.mo_energy[4] * HARTREE_TO_EV
print(f"Ionisation energy: {sto_3g_ie: 0.6f} eV")

Ionisation energy:  14.120110 eV


In [31]:
rhf_mol.analyze()

**** MO energy ****
MO #1   energy= -11.0300340216212  occ= 2
MO #2   energy= -0.910011968065515 occ= 2
MO #3   energy= -0.518904487553552 occ= 2
MO #4   energy= -0.51890448755355  occ= 2
MO #5   energy= -0.51890448755355  occ= 2
MO #6   energy= 0.715643667569359  occ= 0
MO #7   energy= 0.715643667569361  occ= 0
MO #8   energy= 0.715643667569362  occ= 0
MO #9   energy= 0.75495845102494   occ= 0
 ** Mulliken atomic charges  **
charge of    0C =     -0.15915
charge of    1H =      0.03979
charge of    2H =      0.03979
charge of    3H =      0.03979
charge of    4H =      0.03979
Dipole moment(X, Y, Z, Debye):  0.00000,  0.00000, -0.00000


((array([1.99977571, 1.00565083, 1.05123967, 1.05123967, 1.05123967,
         0.96021362, 0.96021362, 0.96021362, 0.96021362]),
  array([-0.15914554,  0.03978638,  0.03978638,  0.03978638,  0.03978638])),
 array([ 1.36862412e-14,  3.10409593e-15, -8.32462091e-15]))